In [2]:
with open('hostel_bois.txt', 'r', encoding='utf-8') as f:
 content = f.read()

In [3]:
lines = content.split('\n')

In [4]:
with open('/content/hostel_bois.txt', 'r', encoding='utf-8') as f:
 content

# Feature 1 - The Chat Parser

In [5]:
# List to store all messages
messages = []

# Counters
system_count = 0
media_count = 0
deleted_count = 0

# Open the file
file = open("hostel_bois.txt", "r", encoding="utf-8")

for line in file:

    # Remove newline character
    line = line.replace("\n", "")

    # Skip empty lines
    if line == "":
        continue

    # Split timestamp and remaining part
    parts = line.split(" - ", 1)

    # Skip invalid lines
    if len(parts) != 2:
        continue

    timestamp = parts[0]
    rest = parts[1]

    # Check for system messages
    if ": " not in rest:
        system_count += 1
        continue

    # Split sender and message
    sender, message = rest.split(": ", 1)

    # Count media messages
    if message == "<Media omitted>":
        media_count += 1

    # Count deleted messages
    elif message == "This message was deleted":
        deleted_count += 1

    # Store data in dictionary
    message_data = {
        "timestamp": timestamp,
        "sender": sender,
        "message": message
    }

    messages.append(message_data)

file.close()

# Display results
print("Feature 1 Completed")
print("---------------------------")
print("Parsed Messages :", len(messages))
print("System Messages :", system_count)
print("Media Messages  :", media_count)
print("Deleted Messages:", deleted_count)

Feature 1 Completed
---------------------------
Parsed Messages : 3174
System Messages : 4
Media Messages  : 32
Deleted Messages: 15


# Feature 2 - Group Overview

In [6]:
# Open the chat file
file = open("hostel_bois.txt", "r", encoding="utf-8")

# Dictionary to store message count
person_count = {}

# Set to store unique participants
participants = set()

# Read the file line by line
for line in file:

    # Remove newline character
    line = line.replace("\n", "")

    # Skip empty lines
    if line == "":
        continue

    # Split timestamp and remaining text
    parts = line.split(" - ", 1)

    # Skip invalid lines
    if len(parts) != 2:
        continue

    timestamp = parts[0]
    rest = parts[1]

    # Skip system messages
    if ": " not in rest:
        continue

    # Split sender and message
    sender, message = rest.split(": ", 1)

    # Add sender to the set
    participants.add(sender)

    # Count messages
    if sender in person_count:
        person_count[sender] += 1
    else:
        person_count[sender] = 1

file.close()

# Convert dictionary into a list
people = list(person_count.items())

# Sort the list in descending order of message count
for i in range(len(people)):
    for j in range(i + 1, len(people)):
        if people[i][1] < people[j][1]:
            temp = people[i]
            people[i] = people[j]
            people[j] = temp

# Get first and last dates
first_date = messages[0]["timestamp"].split(",")[0]
last_date = messages[len(messages) - 1]["timestamp"].split(",")[0]

# Print Group Overview
print("==========================================")
print("            GROUP OVERVIEW")
print("==========================================")

print("Total Messages :", len(messages))
print(f"Date Range     : {first_date} to {last_date}")
print("Participants   :", len(participants))

print("Messages Per Person")
print("------------------------------")

# Print message count and percentage
for person, count in people:

    percentage = (count / len(messages)) * 100

    print(f"{person} : {count} ({percentage:.1f}%)")

            GROUP OVERVIEW
Total Messages : 3174
Date Range     : 01/04/24 to 30/05/24
Participants   : 6
Messages Per Person
------------------------------
Rahul : 953 (30.0%)
Priya : 718 (22.6%)
Neha : 635 (20.0%)
Aman : 490 (15.4%)
Karan : 354 (11.2%)
Vikas : 24 (0.8%)


# Feature 3 - Most Active Day and Hour



In [7]:
# Open the chat file
file = open("hostel_bois.txt", "r", encoding="utf-8")

# Dictionaries to count messages
day_count = {}
hour_count = {}

# Read the file
for line in file:

    # Remove newline character
    line = line.replace("\n", "")

    # Skip empty lines
    if line == "":
        continue

    # Split timestamp and remaining text
    parts = line.split(" - ", 1)

    # Skip invalid lines
    if len(parts) != 2:
        continue

    timestamp = parts[0]
    rest = parts[1]

    # Skip system messages
    if ": " not in rest:
        continue

    # Get date
    date = timestamp.split(",")[0]

    # Get hour
    time = timestamp.split(",")[1]
    hour = time.split(":")[0].replace(" ", "")

    # Count messages for each day
    if date in day_count:
        day_count[date] += 1
    else:
        day_count[date] = 1

    # Count messages for each hour
    if hour in hour_count:
        hour_count[hour] += 1
    else:
        hour_count[hour] = 1

file.close()

# Find busiest day
max_day = ""
max_day_messages = 0

for day in day_count:
    if day_count[day] > max_day_messages:
        max_day_messages = day_count[day]
        max_day = day

# Find busiest hour
max_hour = ""
max_hour_messages = 0

for hour in hour_count:
    if hour_count[hour] > max_hour_messages:
        max_hour_messages = hour_count[hour]
        max_hour = hour

# Print results
print("========================================")
print("     MOST ACTIVE DAY AND HOUR")
print("========================================")

print("Busiest Day  :", max_day)
print("Messages     :", max_day_messages)

print()

print("Busiest Hour :", max_hour + ":00 - " + max_hour + ":59")
print("Messages     :", max_hour_messages)

     MOST ACTIVE DAY AND HOUR
Busiest Day  : 04/05/24
Messages     : 76

Busiest Hour : 18:00 - 18:59
Messages     : 248


# Feature 4 - Activity Heatmap (NumPy)


In [8]:
import numpy as np

# Get participant names
participants = sorted(set(msg["sender"] for msg in messages))

# Create a dictionary to map participant to row index
person_index = {}

for i in range(len(participants)):
    person_index[participants[i]] = i

# Create 6 x 24 matrix filled with zeros
heatmap = np.zeros((len(participants), 24), dtype=int)

# Fill the matrix
for msg in messages:

    sender = msg["sender"]

    # Extract hour
    time = msg["timestamp"].split(",")[1].strip()
    hour = int(time.split(":")[0])

    row = person_index[sender]

    heatmap[row][hour] += 1

# Print the matrix
print("=" * 90)
print("ACTIVITY HEATMAP (Messages per Hour)")
print("=" * 90)

print("Hour -> ", end="")
for h in range(24):
    print(f"{h:3}", end=" ")
print()

for i in range(len(participants)):
    print(f"{participants[i]:8}", end=" ")
    for j in range(24):
        print(f"{heatmap[i][j]:3}", end=" ")
    print()

ACTIVITY HEATMAP (Messages per Hour)
Hour ->   0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23 
Aman      54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7  16   8  13  11   0  56 
Karan      0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27  25  32  23  14   9   8 
Neha       0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47  62  50  45  27  28  30 
Priya      0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40  38  60  43  32  18   9 
Rahul      3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49 105  76  41  92  60  54 
Vikas      0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3   2   2   1   1   1   2 


# Feature 5 - Top Words

In [13]:
word_count = {}

# Common words to ignore
stop_words = {
    "the", "is", "are", "a", "an", "to", "of", "and", "in",
    "on", "for", "it", "this", "that", "i", "you", "we",
    "he", "she", "they", "my", "your", "our",
    "hai", "haan", "yaar", "bhai", "na", "ki", "ka", "ke",
    "me", "ho", "hi", "ok", "okay", "lol", "so", "am"
}

for msg in messages:

    text = msg["message"]

    # Skip media and deleted messages
    if text == "<Media omitted>" or text == "This message was deleted":
        continue

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    for ch in ".,!?;:()[]{}<>\"'/":
        text = text.replace(ch, "")

    # Split into words
    words = text.split()

    # Count words
    for word in words:

        if word not in stop_words:

            if word in word_count:
                word_count[word] += 1
            else:
                word_count[word] = 1

# Sort words by frequency
top_words = sorted(word_count.items(),
                   key=lambda x: x[1],
                   reverse=True)

print("=" * 40)
print("TOP 5 MOST USED WORDS")
print("=" * 40)

 # Display top 5 words
for word, count in top_words[:5]:

        # Create text bar
        bar = "█" * (count // 10)

        print(f"{word:<12} {bar} {count}")

TOP 5 MOST USED WORDS
was          ██████████████████████████████████████ 385
how          ████████████████████████████████ 321
guys         ███████████████████████████████ 318
about        ███████████████████████████ 274
today        █████████████████████████ 257


# Feature 6 - Response Speed & Silent Streaks


In [14]:
from datetime import datetime

def response_patterns(messages):

    # Dictionary to store total reply time (in minutes)
    total_time = {}

    # Dictionary to store number of replies
    reply_count = {}

    # Compare consecutive messages
    for i in range(1, len(messages)):

        previous = messages[i - 1]
        current = messages[i]

        # Ignore if same person sent both messages
        if previous["sender"] == current["sender"]:
            continue

        # Convert timestamps to datetime
        t1 = datetime.strptime(previous["timestamp"], "%d/%m/%y, %H:%M")
        t2 = datetime.strptime(current["timestamp"], "%d/%m/%y, %H:%M")

        # Reply time in minutes
        minutes = (t2 - t1).total_seconds() / 60

        sender = current["sender"]

        # Store reply time
        if sender not in total_time:
            total_time[sender] = minutes
            reply_count[sender] = 1
        else:
            total_time[sender] += minutes
            reply_count[sender] += 1

    # Find average reply time
    average = {}

    for person in total_time:
        average[person] = total_time[person] / reply_count[person]

    # Sort by average reply time
    result = sorted(average.items(), key=lambda x: x[1])

    fastest = result[0]
    slowest = result[-1]

    print("======================================")
    print("       RESPONSE PATTERNS")
    print("======================================")

    print(f"Fastest Replier : {fastest[0]} (avg {fastest[1]:.1f} minutes)")

    if slowest[1] >= 60:
        print(f"Slowest Replier : {slowest[0]} (avg {slowest[1]/60:.1f} hours)")
    else:
        print(f"Slowest Replier : {slowest[0]} (avg {slowest[1]:.1f} minutes)")


# Call the function
response_patterns(messages)

       RESPONSE PATTERNS
Fastest Replier : Rahul (avg 34.9 minutes)
Slowest Replier : Aman (avg 55.4 minutes)


In [16]:
from datetime import datetime, timedelta

# Dictionary to store the dates on which each person sent messages
person_dates = {}

# Get all participants
for msg in messages:

    sender = msg["sender"]

    date = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M").date()

    if sender not in person_dates:
        person_dates[sender] = []

    if date not in person_dates[sender]:
        person_dates[sender].append(date)

# Get first and last date of the chat
start_date = datetime.strptime(messages[0]["timestamp"], "%d/%m/%y, %H:%M").date()
end_date = datetime.strptime(messages[-1]["timestamp"], "%d/%m/%y, %H:%M").date()

print("\n==========================================")
print("LONGEST SILENT STREAKS")
print("==========================================")

for person in sorted(person_dates):

    dates = sorted(person_dates[person])

    longest = 0
    current = 0

    longest_start = None
    longest_end = None

    previous = start_date - timedelta(days=1)

    for date in dates:

        gap = (date - previous).days - 1

        if gap > longest:
            longest = gap
            longest_start = previous + timedelta(days=1)
            longest_end = date - timedelta(days=1)

        previous = date

    # Check silence after the last message
    gap = (end_date - previous).days

    if gap > longest:
        longest = gap
        longest_start = previous + timedelta(days=1)
        longest_end = end_date

    # Print result
    if longest == 0:
        print(f"{person:<10}: 0 days (never went silent)")
    elif longest == 1:
        print(f"{person:<10}: 1 day")
    else:
        print(f"{person:<10}: {longest} days ({longest_start.strftime('%d %b')} to {longest_end.strftime('%d %b')})")


LONGEST SILENT STREAKS
Aman      : 0 days (never went silent)
Karan     : 0 days (never went silent)
Neha      : 0 days (never went silent)
Priya     : 0 days (never went silent)
Rahul     : 0 days (never went silent)
Vikas     : 11 days (23 Apr to 03 May)


# Feature 7 - Personality Archetype Detection

In [ ]:
msg_count = {}
night_count = {}
word_count = {}
care_count = {}
drama_count = {}
active_days = {}

burst_sum = {}
burst_count = {}

care_words = ["eat", "sleep", "care", "please", "safe",
              "drink", "water", "reminder", "okay"]

prev_sender = ""
burst = 0

for msg in messages:

    sender = msg["sender"]
    text = msg["message"]
    timestamp = msg["timestamp"]

    if sender not in msg_count:
        msg_count[sender] = 0
        night_count[sender] = 0
        word_count[sender] = 0
        care_count[sender] = 0
        drama_count[sender] = 0
        active_days[sender] = set()
        burst_sum[sender] = 0
        burst_count[sender] = 0

    msg_count[sender] += 1

    date, time = timestamp.split(", ")
    hour = int(time.split(":")[0])

    active_days[sender].add(date)

    # Night Owl
    if hour >= 23 or hour <= 4:
        night_count[sender] += 1

    # Storyteller
    word_count[sender] += len(text.split())

    # Group Mom
    text = text.lower()
    for word in care_words:
        if word in text:
            care_count[sender] += 1

    # Drama Queen
    if msg["message"].isupper() or msg["message"].count("!") >= 2:
        drama_count[sender] += 1

    # Spammer
    if sender == prev_sender:
        burst += 1
    else:
        if prev_sender != "":
            burst_sum[prev_sender] += burst
            burst_count[prev_sender] += 1
        burst = 1

    prev_sender = sender

# Last burst
burst_sum[prev_sender] += burst
burst_count[prev_sender] += 1

# Spammer
spammer = max(burst_sum,
              key=lambda x: burst_sum[x] / burst_count[x])

# Group Mom
group_mom = max(care_count, key=care_count.get)

# Night Owl
night_owl = max(night_count,
                key=lambda x: night_count[x] / msg_count[x])

# Storyteller
storyteller = max(word_count,
                  key=lambda x: word_count[x] / msg_count[x])

# Drama Queen
drama_queen = max(drama_count,
                  key=lambda x: drama_count[x] / msg_count[x])

# Ghost
ghost = min(active_days,
            key=lambda x: len(active_days[x]))

print("=" * 40)
print("PERSONALITY ARCHETYPE DETECTION")
print("=" * 40)
print(spammer, "-> THE SPAMMER")
print(group_mom, "-> THE GROUP MOM")
print(night_owl, "-> THE NIGHT OWL")
print(storyteller, "-> THE STORYTELLER")
print(drama_queen, "-> THE DRAMA QUEEN")
print(ghost, "-> THE GHOST")

PERSONALITY ARCHETYPE DETECTION
Rahul -> THE SPAMMER
Priya -> THE GROUP MOM
Aman -> THE NIGHT OWL
Karan -> THE STORYTELLER
Neha -> THE DRAMA QUEEN
Vikas -> THE GHOST


# Final Report



```
==========================================================================================================================
                 GROUPDNA REPORT - "Hostel Bois"
       60 days  •  3,174 messages  •  6 members
==========================================================================================================================

Period         : 01 April 2024 to 30 May 2024
Busiest day    : 04 May 2024   (76 messages)
Busiest hour   : 18:00 - 18:59 (248 messages)

MESSAGES PER PERSON
   Rahul   ██████████████████████████████ 953  (30.0%)
   Priya   ███████████████████████        718  (22.6%)
   Neha    ████████████████████           635  (20.0%)
   Aman    ███████████████                490  (15.4%)
   Karan   ███████████                    354  (11.2%)
   Vikas   █                              24  (0.8%)

=========================================================================================================================
ACTIVITY HEATMAP (Messages per Hour)
=========================================================================================================================

Hour ->    00  01  02  03  04  05  06  07  08  09  10  11  12  13  14  15  16  17  18  19  20  21  22  23

Aman       54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7  16   8  13  11   0  56
Karan       0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27  25  32  23  14   9   8
Neha        0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47  62  50  45  27  28  30
Priya       0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40  38  60  43  32  18   9
Rahul       3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49 105  76  41  92  60  54
Vikas       0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3   2   2   1   1   1   2

=========================================================================================================================
THIS GROUP'S FAVOURITE WORDS
=========================================================================================================================

   was          ██████████████████████████████████████ 385
   how          ████████████████████████████████       321
   guys         ███████████████████████████████        318
   about        ███████████████████████████            274
   today        █████████████████████████              257

=========================================================================================================================
RESPONSE PATTERNS

   Fastest replier : Rahul (avg 34.9 minutes)
   Slowest replier : Aman  (avg 55.4 minutes)

=========================================================================================================================
LONGEST SILENT STREAKS
=========================================================================================================================

   Aman    : 0 days (never went silent)
   Karan   : 0 days (never went silent)
   Neha    : 0 days (never went silent)
   Priya   : 0 days (never went silent)
   Rahul   : 0 days (never went silent)
   Vikas   : 11 days (23 Apr to 03 May)

=========================================================================================================================
PERSONALITY ARCHETYPES
=========================================================================================================================

   Rahul   -> THE SPAMMER
   Priya   -> THE GROUP MOM
   Aman    -> THE NIGHT OWL
   Karan   -> THE STORYTELLER
   Neha    -> THE DRAMA QUEEN
   Vikas   -> THE GHOST

=========================================================================================================================
Generated by GroupDNA • Built with Python + NumPy
=========================================================================================================================


```

